# Tahap 3: Data Ingestion & SQL Interview Mastery (`load_to_db`)
## Proyek: Indonesian Data Job Market Intelligence

Notebook ini memiliki dua tujuan utama:
1. **Engineering (Data Ingestion & Relational Modeling):** Membangun skema relasional, indeks, dan views di basis data **PostgreSQL** (`loker_data`), lalu memuat data master bersih (`jobs_master` dan `skills_master`).
2. **Analytics & Portfolio Mastery:** Menerapkan seluruh konsep dari panduan **`markdown/Konsep SQL untuk Interview Data Analyst Junior.md`** secara komprehensif menggunakan data riil lowongan kerja data di Indonesia.

--- 
## BAGIAN I: DATA INGESTION & RELATIONAL MODELING
--- 
### 1. Inisialisasi Environment & Konfigurasi Path
Mempersiapkan dependensi Python, konfigurasi path direktori proyek, dan memuat variabel lingkungan dari `.env`.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, inspect

# Menentukan path direktori proyek
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent.parent if CURRENT_DIR.name == "load_data" else CURRENT_DIR

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SQL_DIR = PROJECT_ROOT / "sql"
ENV_PATH = PROJECT_ROOT / ".env"

# Memuat kredensial dari .env
load_dotenv(ENV_PATH)

print(f"Project Root       : {PROJECT_ROOT}")
print(f"Data Processed Dir : {DATA_PROCESSED_DIR}")
print(f"SQL Directory      : {SQL_DIR}")
print(f".env File Exists   : {ENV_PATH.exists()}")

Project Root       : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia
Data Processed Dir : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\processed
SQL Directory      : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\sql
.env File Exists   : True


### 2. Koneksi ke Database PostgreSQL
Membangun koneksi ke basis data `loker_data` menggunakan SQLAlchemy engine dengan driver `psycopg2`.

In [2]:
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "loker_data")

CONNECTION_STRING = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(CONNECTION_STRING, pool_pre_ping=True)

try:
    with engine.connect() as conn:
        version_info = conn.execute(text("SELECT version();")).scalar()
        current_db = conn.execute(text("SELECT current_database();")).scalar()
        current_user = conn.execute(text("SELECT current_user;")).scalar()
        print("Koneksi ke PostgreSQL Berhasil!")
        print(f"   Database     : {current_db}")
        print(f"   User         : {current_user}")
        print(f"   Host / Port  : {DB_HOST}:{DB_PORT}")
        print(f"   Server Info  : {version_info.split(',')[0]}")
except Exception as e:
    print(f"Gagal terkoneksi ke PostgreSQL: {e}")
    raise

Koneksi ke PostgreSQL Berhasil!
   Database     : loker_data
   User         : postgres
   Host / Port  : localhost:5432
   Server Info  : PostgreSQL 17.2 on x86_64-windows


### 3. Eksekusi Skema DDL & Analytical Views (`sql/schema.sql` & `sql/views.sql`)
Mengeksekusi pembuatan tabel relasional (`jobs_master`, `skills_master`), primary & foreign key constraints, indeks performa, dan SQL views.

In [3]:
def execute_sql_file(engine, file_path):
    if not file_path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}")
    with open(file_path, "r", encoding="utf-8") as f:
        sql_content = f.read()
    statements = [stmt.strip() for stmt in sql_content.split(";") if stmt.strip()]
    with engine.begin() as conn:
        for stmt in statements:
            conn.execute(text(stmt))
    print(f"Berhasil mengeksekusi {len(statements)} statement dari {file_path.name}")

# Eksekusi Schema DDL & Analytical Views
execute_sql_file(engine, SQL_DIR / "schema.sql")
execute_sql_file(engine, SQL_DIR / "views.sql")

inspector = inspect(engine)
tables = inspector.get_table_names(schema="public")
views = inspector.get_view_names(schema="public")
print(f"\nTabel terdaftar di schema 'public' : {tables}")
print(f"Views terdaftar di schema 'public' : {views}")

Berhasil mengeksekusi 32 statement dari schema.sql
Berhasil mengeksekusi 4 statement dari views.sql

Tabel terdaftar di schema 'public' : ['jobs_master', 'skills_master']
Views terdaftar di schema 'public' : ['v_role_salary_summary', 'v_top_skills_overall', 'v_skills_by_role', 'v_industry_role_summary']


### 4. Pemuatan Data Master ke Database (ETL Data Ingestion)
Membaca dataset master terstandarisasi (`.parquet` / `.csv`) dan memuatnya ke tabel PostgreSQL.

In [4]:
jobs_parquet = DATA_PROCESSED_DIR / "jobs_master.parquet"
skills_parquet = DATA_PROCESSED_DIR / "skills_master.parquet"

# Baca jobs_master
df_jobs = pd.read_parquet(jobs_parquet) if jobs_parquet.exists() else pd.read_csv(DATA_PROCESSED_DIR / "jobs_master.csv")
print(f"Membaca jobs_master   : {df_jobs.shape[0]} baris, {df_jobs.shape[1]} kolom")

# Baca skills_master
df_skills = pd.read_parquet(skills_parquet) if skills_parquet.exists() else pd.read_csv(DATA_PROCESSED_DIR / "skills_master.csv")
print(f"Membaca skills_master : {df_skills.shape[0]} baris, {df_skills.shape[1]} kolom")

# Format datetime
df_jobs["collected_at"] = pd.to_datetime(df_jobs["collected_at"], errors="coerce")

# Pemuatan ke jobs_master (Parent)
df_jobs.to_sql(name="jobs_master", con=engine, if_exists="append", index=False, method="multi", chunksize=500)
print("Data jobs_master berhasil dimuat!")

# Pemuatan ke skills_master (Child)
df_skills.to_sql(name="skills_master", con=engine, if_exists="append", index=False, method="multi", chunksize=500)
print("Data skills_master berhasil dimuat!")

Membaca jobs_master   : 250 baris, 23 kolom
Membaca skills_master : 1061 baris, 5 kolom
Data jobs_master berhasil dimuat!


Data skills_master berhasil dimuat!


### 5. Verifikasi Integritas Data & Quality Gate
Memastikan konsistensi jumlah baris, validitas Foreign Key, dan keunikan Primary Key.

In [5]:
with engine.connect() as conn:
    db_jobs_count = conn.execute(text("SELECT COUNT(*) FROM jobs_master;")).scalar()
    db_skills_count = conn.execute(text("SELECT COUNT(*) FROM skills_master;")).scalar()
    fk_orphans = conn.execute(text("""
        SELECT COUNT(*)
        FROM skills_master s
        LEFT JOIN jobs_master j ON s.job_id = j.job_id
        WHERE j.job_id IS NULL;
    """)).scalar()
    pk_duplicates = conn.execute(text("""
        SELECT COUNT(*)
        FROM (
            SELECT job_id, COUNT(*)
            FROM jobs_master
            GROUP BY job_id
            HAVING COUNT(*) > 1
        ) dup;
    """)).scalar()

status_jobs = 'PASS' if db_jobs_count == len(df_jobs) else 'FAIL'
status_skills = 'PASS' if db_skills_count == len(df_skills) else 'FAIL'
status_fk = 'PASS' if fk_orphans == 0 else 'FAIL'
status_pk = 'PASS' if pk_duplicates == 0 else 'FAIL'

print("=" * 60)
print("HASIL VALIDASI INTEGRITAS BASIS DATA:")
print("=" * 60)
print(f"1. jobs_master row count   : {db_jobs_count} baris (Pandas: {len(df_jobs)}) -> {status_jobs}")
print(f"2. skills_master row count : {db_skills_count} baris (Pandas: {len(df_skills)}) -> {status_skills}")
print(f"3. Foreign Key orphans     : {fk_orphans} orphan records -> {status_fk}")
print(f"4. Primary Key duplicates  : {pk_duplicates} duplicates -> {status_pk}")
print("=" * 60)

HASIL VALIDASI INTEGRITAS BASIS DATA:
1. jobs_master row count   : 250 baris (Pandas: 250) -> PASS
2. skills_master row count : 1061 baris (Pandas: 1061) -> PASS
3. Foreign Key orphans     : 0 orphan records -> PASS
4. Primary Key duplicates  : 0 duplicates -> PASS


--- 
## BAGIAN II: PENERAPAN KONSEP SQL UNTUK INTERVIEW DATA ANALYST JUNIOR
Mengacu pada panduan **`markdown/Konsep SQL untuk Interview Data Analyst Junior.md`**, bagian ini mempraktikkan langsung konsep-konsep kunci dan soal-soal interview populer menggunakan data riil lowongan kerja.
--- 
### Konsep 1: Dasar Query & Operator Filtering
**Elemen:** `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN`, `BETWEEN`, `LIKE/ILIKE`, `AND/OR/NOT`, `AS`.

**Studi Kasus:** Temukan 10 lowongan bidang *Data Analytics & BI* di pulau Jawa dengan estimasi gaji antara Rp 7.000.000 - Rp 15.000.000 yang memiliki kata kunci 'Analyst' atau 'Analis'.

In [6]:
q_dasar = text("""
SELECT 
    job_id AS id_lowongan,
    title_raw AS judul_posisi,
    company_raw AS perusahaan,
    province AS provinsi,
    city AS kota,
    salary_median AS estimasi_gaji
FROM jobs_master
WHERE role_group = 'Data Analytics & BI'
  AND province IN ('DKI Jakarta', 'Jawa Barat', 'Jawa Tengah', 'Jawa Timur')
  AND salary_median BETWEEN 7000000 AND 15000000
  AND (title_raw ILIKE '%analyst%' OR title_raw ILIKE '%analis%')
ORDER BY salary_median DESC, title_raw ASC
LIMIT 10;
""")

with engine.connect() as conn:
    df_q1 = pd.read_sql(q_dasar, conn)
display(df_q1)

,id_lowongan,judul_posisi,perusahaan,provinsi,kota,estimasi_gaji
0,JBS_95292658,Risk Analyst,PT Dupoin Group Indonesia,DKI Jakarta,Jakarta Selatan,11250000.0
1,JBS_95492157,Commercial Business Analyst,PT DUNIA KECANTIKAN INDONESIA,DKI Jakarta,Jakarta Pusat,10000000.0
2,GLN_aa07c9b62ee94dccbbbc64ff0734a98f,Data Analyst,PT Sigma Global Teknologi,DKI Jakarta,Jakarta Selatan,10000000.0
3,GLN_06ecc90733644f7096bad04eeff207f2,Data Analyst,PT Pro Sigmaka Mandiri,DKI Jakarta,Jakarta Selatan,9000000.0
4,GLN_ab5e3448cb5a4ba4ac3db30a98a02335,Business Analyst,PT. Solusi Kode Indonesia,DKI Jakarta,Jakarta Selatan,8500000.0
5,GLN_2bc42fe49b954dacba7f5badbe2433ec,Business Data Analyst (Accounting Divison),Nusantara Sakti Group,DKI Jakarta,Jakarta Barat,8500000.0
6,GLN_a072678371244916879752a6bee58e07,Data Analyst,Nusantara Sakti Group,Jawa Tengah,Semarang,8500000.0
7,GLN_9a6f1864e2a34916b0729160edcb851f,Data Analyst,Nusantara Sakti Group,DKI Jakarta,Jakarta Barat,8500000.0
8,GLN_427fafa61271458590703a92dd8deaab,Data Analyst,Nusantara Sakti Group,Jawa Timur,Surabaya,8500000.0
9,JBS_95503060,Finance Data Analyst,PT Nusantara Sakti Group (JAKARTA),DKI Jakarta,Jakarta Barat,8500000.0


### Konsep 2: Agregasi, GROUP BY vs HAVING, & Perilaku NULL
**Poin Kunci Interview:**
1. **`WHERE` vs `HAVING`:** `WHERE` memfilter baris *sebelum* pengelompokan; `HAVING` memfilter hasil agregat *setelah* `GROUP BY`.
2. **`COUNT(*)` vs `COUNT(kolom)`:** `COUNT(kolom)` **mengabaikan NULL**, sedangkan `COUNT(*)` menghitung seluruh baris fisik.
3. **Jebakan `AVG()` pada data dengan NULL:** Perhatikan perbedaan antara rata-rata dari data yang mempublikasikan gaji vs rata-rata jika NULL diubah jadi 0 menggunakan `COALESCE()`.

In [7]:
q_agregasi = text("""
SELECT 
    role_group,
    COUNT(*) AS total_lowongan_fisik,
    COUNT(salary_median) AS lowongan_ada_gaji,
    COUNT(*) - COUNT(salary_median) AS lowongan_gaji_null,
    ROUND(AVG(salary_median), 0) AS rerata_gaji_valid,
    ROUND(AVG(COALESCE(salary_median, 0)), 0) AS rerata_jika_null_jadi_nol
FROM jobs_master
GROUP BY role_group
HAVING COUNT(*) >= 20
ORDER BY total_lowongan_fisik DESC;
""")

with engine.connect() as conn:
    df_q2 = pd.read_sql(q_agregasi, conn)
display(df_q2)

,role_group,total_lowongan_fisik,lowongan_ada_gaji,lowongan_gaji_null,rerata_gaji_valid,rerata_jika_null_jadi_nol
0,Data Analytics & BI,135,63,72,6347861.0,2962335.0
1,Data Engineering,50,25,25,8626000.0,4313000.0
2,Machine Learning & AI,43,29,14,10110552.0,6818744.0
3,Data Science,22,8,14,7625000.0,2772727.0


> [!NOTE]
> Perhatikan kolom `rerata_jika_null_jadi_nol` nilainya anjlok drastis dibanding `rerata_gaji_valid`. Dalam interview, jelaskan selalu asumsi penanganan nilai NULL sebelum menghitung rata-rata!

### Konsep 3: Jenis-Jenis JOIN & Kasus Anti-Join (Interview Favorite)
**Poin Kunci Interview:**
- **`INNER JOIN`**: Mengambil baris yang berelasi di kedua tabel.
- **`LEFT JOIN`**: Mengambil semua baris tabel kiri; jika tidak ada padanan di tabel kanan, diisi `NULL`.
- **Anti-Join (`LEFT JOIN + WHERE ... IS NULL`)**: Pertanyaan wajib interview untuk kasus *"Cari entitas di tabel A yang tidak pernah memiliki transaksi di tabel B"* (misal lowongan tanpa keahlian teknis terdaftar).

In [8]:
q_antijoin = text("""
SELECT 
    j.job_id,
    j.title_raw,
    j.company_raw,
    j.role_group,
    s.skill_name
FROM jobs_master j
LEFT JOIN skills_master s ON j.job_id = s.job_id
WHERE s.job_id IS NULL
LIMIT 5;
""")

with engine.connect() as conn:
    df_antijoin = pd.read_sql(q_antijoin, conn)

print("Contoh Hasil Anti-Join (Lowongan yang tidak memiliki entri skills di skills_master):")
display(df_antijoin)

Contoh Hasil Anti-Join (Lowongan yang tidak memiliki entri skills di skills_master):


,job_id,title_raw,company_raw,role_group,skill_name
0,GLN_07401ee8bc95483ca0baba1779d88770,Data Analyst,PT Sigma Digital Nusantara,Data Analytics & BI,None
1,GLN_7c091974ce4048d6a197f457d52acd9f,Data Analyst,Fajar Indah Knalpot,Data Analytics & BI,None
2,GLN_69a2d926d80d4c5baea97c9c26570c9e,Data Analyst,Aster Group,Data Analytics & BI,None
3,GLN_4fd2df41ed9649c29314f118dc16196e,Data Analyst (MagangHub),Redigma,Data Analytics & BI,None
4,GLN_38f0c82f11054c74b2364c223638cc74,Staff Data Analyst,CV. Indo Sejahtera Abadi,Data Analytics & BI,None


### Konsep 4: Penanganan NULL & Logika Kondisional (`COALESCE`, `NULLIF`, `CASE WHEN`)
**Poin Kunci Interview:**
- `COALESCE(val, fallback)`: Mengembalikan nilai non-null pertama.
- `NULLIF(val1, val2)`: Mengembalikan `NULL` jika kedua nilai sama (berguna untuk mencegah division by zero atau mendeteksi nilai kembar).
- `CASE WHEN`: Untuk kategorisasi kustom dan **Conditional Aggregation**.

In [9]:
q_null_case = text("""
SELECT 
    industry_category,
    COUNT(*) AS total_jobs,
    -- Conditional Aggregation:
    COUNT(CASE WHEN work_mode = 'On-site' THEN 1 END) AS on_site_count,
    COUNT(CASE WHEN work_mode = 'Hybrid' THEN 1 END) AS hybrid_count,
    COUNT(CASE WHEN work_mode = 'Remote' THEN 1 END) AS remote_count,
    ROUND(COUNT(CASE WHEN work_mode = 'Remote' THEN 1 END)::NUMERIC / COUNT(*) * 100, 1) AS remote_pct,
    -- Penanganan NULL:
    COALESCE(ROUND(AVG(salary_median), 0)::TEXT, 'Dirahasiakan') AS rerata_gaji_disclose
FROM jobs_master
GROUP BY industry_category
ORDER BY total_jobs DESC;
""")

with engine.connect() as conn:
    df_null_case = pd.read_sql(q_null_case, conn)
display(df_null_case)

,industry_category,total_jobs,on_site_count,hybrid_count,remote_count,remote_pct,rerata_gaji_disclose
0,Lainnya / Tidak Disebutkan,81,77,3,1,1.2,6281411
1,Information Technology & Software,58,51,4,3,5.2,10763818
2,Banking & Financial Services,28,27,1,0,0.0,9134615
3,"Retail, FMCG & E-Commerce",25,25,0,0,0.0,6155000
4,Logistics & Transportation,20,19,1,0,0.0,6213889
5,Manufacturing & Heavy Industry,15,13,2,0,0.0,4172586
6,"Media, Education & Services",14,12,1,1,7.1,6131250
7,Healthcare & Pharmaceuticals,9,7,1,1,11.1,12312500


### Konsep 5: Window Functions (Pembeda Utama Kandidat!)
**Perbedaan Fundamental:**
- **`ROW_NUMBER()`**: Selalu memberikan nomor urut berurutan unik (1, 2, 3, 4, ...).
- **`RANK()`**: Memberikan nomor peringkat yang sama jika ada nilai seri, namun **melompatkan peringkat berikutnya** (contoh: 1, 2, 2, 4).
- **`DENSE_RANK()`**: Memberikan nomor peringkat yang sama jika ada nilai seri, namun **tanpa melompatkan peringkat** (contoh: 1, 2, 2, 3).
- **`LAG()` / `LEAD()`**: Mengakses nilai baris sebelum / sesudahnya tanpa memerlukan self-join.

In [10]:
q_window_comparison = text("""
SELECT 
    company_raw,
    title_raw,
    salary_median,
    ROW_NUMBER() OVER(ORDER BY salary_median DESC NULLS LAST) AS row_num,
    RANK() OVER(ORDER BY salary_median DESC NULLS LAST) AS rank_num,
    DENSE_RANK() OVER(ORDER BY salary_median DESC NULLS LAST) AS dense_rank_num
FROM jobs_master
WHERE is_salary_disclosed = TRUE AND role_group = 'Data Engineering'
LIMIT 8;
""")

with engine.connect() as conn:
    df_win_comp = pd.read_sql(q_window_comparison, conn)

print("Perbandingan Visual ROW_NUMBER vs RANK vs DENSE_RANK:")
display(df_win_comp)

Perbandingan Visual ROW_NUMBER vs RANK vs DENSE_RANK:


,company_raw,title_raw,salary_median,row_num,rank_num,dense_rank_num
0,PT Adecco Personnel Indonesia,Data Engineer - 6 Months Contract (Renewable) ...,27500000.0,1,1,1
1,Insura Media Solusi,Senior Data Engineer,12500000.0,2,2,2
2,PT ROCKSTAR SPORTS SCHOOL,Data Engineer,12500000.0,3,2,2
3,Markethac,Data Engineer,10500000.0,4,4,3
4,PT Sigma Global Teknologi,Data Engineer - Can Join Immediately,10000000.0,5,5,4
5,PT Insura Media Solusi (Jakarta),"Data Engineer (Analyst, ETL, DBA)",10000000.0,6,5,4
6,GITS Indonesia,Junior Data Engineer,9500000.0,7,7,5
7,Kiriminaja,Data Engineer Specialist,9500000.0,8,7,5


> [!TIP]
> Perhatikan baris dengan gaji kembar (Rp 25.000.000): `ROW_NUMBER` memberi nomor urut 2 dan 3, `RANK` memberi ranking 2 dan 2 lalu meloncat ke 4, sedangkan `DENSE_RANK` memberi ranking 2 dan 2 lalu berlanjut ke 3 tanpa melompat.

#### Menggunakan `LAG()` untuk Menghitung Selisih Gaji terhadap Lowongan Tepat di Atasnya

In [11]:
q_lag_lead = text("""
WITH ranked_bi AS (
    SELECT 
        company_raw,
        title_raw,
        salary_median,
        LAG(salary_median) OVER (ORDER BY salary_median DESC) AS higher_salary
    FROM jobs_master
    WHERE is_salary_disclosed = TRUE AND role_group = 'Data Analytics & BI'
    LIMIT 7
)
SELECT 
    company_raw,
    title_raw,
    salary_median,
    higher_salary,
    COALESCE(higher_salary - salary_median, 0) AS salary_gap_down
FROM ranked_bi;
""")

with engine.connect() as conn:
    df_lag = pd.read_sql(q_lag_lead, conn)
display(df_lag)

,company_raw,title_raw,salary_median,higher_salary,salary_gap_down
0,PT Adecco Personnel Indonesia,Business Intelligence 6 Months Contract (Rene...,25000000.0,NaN,0.0
1,Nusantara Sakti Group,Manager Business Data Analyst (Head Office),17500000.0,25000000.0,7500000.0
2,PT Fata Organa Solusi,Business Analyst (Japanese Client Projects),12000000.0,17500000.0,5500000.0
3,PT Dupoin Group Indonesia,Risk Analyst,11250000.0,12000000.0,750000.0
4,PT Beton Perkasa Wijaksana,Kepala Divisi Electronic Data Processing (EDP),11000000.0,11250000.0,250000.0
5,PT DUNIA KECANTIKAN INDONESIA,Commercial Business Analyst,10000000.0,11000000.0,1000000.0
6,PT Sigma Global Teknologi,Data Analyst,10000000.0,10000000.0,0.0


### Konsep 6: Set Operations (`UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`)
**Poin Kunci Interview:**
- `UNION`: Menggabungkan hasil query dan **menghapus duplikat** (lebih berat karena ada sorting/deduplikasi).
- `UNION ALL`: Menggabungkan hasil tanpa menghapus duplikat (lebih cepat).
- `INTERSECT`: Mengambil baris yang ada di **kedua** hasil query (irisan).
- `EXCEPT`: Mengambil baris yang ada di query pertama **tetapi tidak ada** di query kedua.

In [12]:
# 1. INTERSECT: Skill yang dibutuhkan oleh KEDUA role (Data Analyst & Data Engineer)
q_intersect = text("""
SELECT skill_name, 'Dibutuhkan Kedua Role' AS status
FROM skills_master WHERE role_group = 'Data Analytics & BI'
INTERSECT
SELECT skill_name, 'Dibutuhkan Kedua Role' AS status
FROM skills_master WHERE role_group = 'Data Engineering'
ORDER BY skill_name
LIMIT 8;
""")

# 2. EXCEPT: Skill khusus Data Engineering yang TIDAK ada di Data Analytics
q_except = text("""
SELECT skill_name AS skill_khusus_data_engineer
FROM skills_master WHERE role_group = 'Data Engineering'
EXCEPT
SELECT skill_name
FROM skills_master WHERE role_group = 'Data Analytics & BI'
ORDER BY skill_khusus_data_engineer
LIMIT 8;
""")

with engine.connect() as conn:
    print("--- INTERSECT: Keahlian Bersama (Data Analyst & Data Engineer) ---")
    display(pd.read_sql(q_intersect, conn))
    print("\n--- EXCEPT: Keahlian Spesifik Data Engineering Saja ---")
    display(pd.read_sql(q_except, conn))

--- INTERSECT: Keahlian Bersama (Data Analyst & Data Engineer) ---


,skill_name,status
0,AWS,Dibutuhkan Kedua Role
1,Azure,Dibutuhkan Kedua Role
2,Bash / Shell,Dibutuhkan Kedua Role
3,GCP,Dibutuhkan Kedua Role
4,Google Sheets,Dibutuhkan Kedua Role
5,Java,Dibutuhkan Kedua Role
6,Looker,Dibutuhkan Kedua Role
7,Machine Learning,Dibutuhkan Kedua Role



--- EXCEPT: Keahlian Spesifik Data Engineering Saja ---


,skill_khusus_data_engineer
0,Airflow
1,Apache Kafka
2,Apache Spark
3,BigQuery
4,Cassandra
5,ClickHouse
6,Databricks
7,dbt


### Konsep 7: Fungsi Tanggal, String & Konversi Tipe Data (CAST)
Mendemonstrasikan fungsi manipulasi tanggal (`DATE_TRUNC`, `EXTRACT`), manipulasi string (`CONCAT`, `SUBSTRING`, `UPPER`), dan type casting (`::BIGINT`, `CAST`).

In [13]:
q_string_date = text("""
SELECT 
    job_id,
    UPPER(role_group) AS role_group_upper,
    SUBSTRING(company_raw FROM 1 FOR 20) AS company_short,
    CONCAT(city, ', ', province) AS lokasi_lengkap,
    DATE_TRUNC('day', collected_at) AS tgl_scraping,
    EXTRACT(HOUR FROM collected_at) AS jam_scraping,
    CAST(salary_median AS BIGINT) AS gaji_integer
FROM jobs_master
WHERE is_salary_disclosed = TRUE
LIMIT 6;
""")

with engine.connect() as conn:
    df_str_date = pd.read_sql(q_string_date, conn)
display(df_str_date)

,job_id,role_group_upper,company_short,lokasi_lengkap,tgl_scraping,jam_scraping,gaji_integer
0,GLN_a072678371244916879752a6bee58e07,DATA ANALYTICS & BI,Nusantara Sakti Grou,"Semarang, Jawa Tengah",2026-10-31,22.0,8500000
1,GLN_06ecc90733644f7096bad04eeff207f2,DATA ANALYTICS & BI,PT Pro Sigmaka Mandi,"Jakarta Selatan, DKI Jakarta",2026-10-31,22.0,9000000
2,GLN_9a6f1864e2a34916b0729160edcb851f,DATA ANALYTICS & BI,Nusantara Sakti Grou,"Jakarta Barat, DKI Jakarta",2026-10-31,22.0,8500000
3,GLN_49aa60d274704bc6b062d4fc9261dde3,DATA ANALYTICS & BI,PT. Meteor Inovasi D,"Jakarta Selatan, DKI Jakarta",2026-10-31,22.0,7750000
4,GLN_111d465e42264e26b8128eb408d9841c,DATA ANALYTICS & BI,Migoz Group,"Surakarta, Jawa Tengah",2026-10-31,22.0,2800000
5,GLN_aa07c9b62ee94dccbbbc64ff0734a98f,DATA ANALYTICS & BI,PT Sigma Global Tekn,"Jakarta Selatan, DKI Jakarta",2026-10-31,22.0,10000000


--- 
## BAGIAN III: BEDAH 5 SOAL PRAKTIK INTERVIEW TERPOPULER
Di bawah ini adalah 5 variasi soal interview SQL yang paling sering diajukan untuk posisi Junior Data Analyst beserta alur berpikir logisnya.
--- 
### Soal 1: Cari Gaji Tertinggi Kedua (Second Highest Salary)
**Pendekatan:** Menggunakan CTE + `DENSE_RANK()` agar jika terdapat beberapa perusahaan yang sama-sama menawarkan gaji tertinggi pertama, peringkat kedua tetap valid.

In [14]:
q_soal1 = text("""
WITH salary_ranking AS (
    SELECT 
        company_raw,
        title_raw,
        role_group,
        salary_median,
        DENSE_RANK() OVER (ORDER BY salary_median DESC) AS rnk
    FROM jobs_master
    WHERE is_salary_disclosed = TRUE
)
SELECT company_raw, title_raw, role_group, salary_median, rnk
FROM salary_ranking
WHERE rnk = 2;
""")

with engine.connect() as conn:
    df_soal1 = pd.read_sql(q_soal1, conn)
display(df_soal1)

,company_raw,title_raw,role_group,salary_median,rnk
0,PT Adecco Personnel Indonesia,Data Engineer - 6 Months Contract (Renewable) ...,Data Engineering,27500000.0,2


### Soal 2: Temukan Data Duplikat / Multiple Postings
**Kasus Bisnis:** *"Apakah ada perusahaan yang memposting lowongan berulang untuk posisi dan kelompok profesi yang sama?"*
**Pendekatan:** `GROUP BY` pada kolom penentu, lalu filter menggunakan `HAVING COUNT(*) > 1`.

In [15]:
q_soal2 = text("""
SELECT 
    company_raw,
    title_raw,
    role_group,
    COUNT(*) AS total_postings
FROM jobs_master
GROUP BY company_raw, title_raw, role_group
HAVING COUNT(*) > 1
ORDER BY total_postings DESC;
""")

with engine.connect() as conn:
    df_soal2 = pd.read_sql(q_soal2, conn)
display(df_soal2)

,company_raw,title_raw,role_group,total_postings
0,Nusantara Sakti Group,Data Analyst,Data Analytics & BI,3
1,PT Sigma Global Teknologi,Data Engineer - Can Join Immediately,Data Engineering,2
2,PT. Pancasona Dayasakti,Sales Data Analyst,Data Analytics & BI,2
3,PT Sigma Global Teknologi,Data Analyst,Data Analytics & BI,2
4,PT Sigma Global Teknologi,Data Engineer,Data Engineering,2


### Soal 3: Top N per Kategori (Top 3 Keahlian Paling Dicari per Profesi)
**Pendekatan:** Agregasi frekuensi skill per `role_group`, lalu gunakan Window Function `DENSE_RANK() OVER(PARTITION BY role_group ORDER BY count DESC)` dan filter `rank <= 3` pada outer query.

In [16]:
q_soal3 = text("""
WITH skill_ranks AS (
    SELECT 
        role_group,
        skill_name,
        skill_category,
        COUNT(DISTINCT job_id) AS demand_count,
        DENSE_RANK() OVER(PARTITION BY role_group ORDER BY COUNT(DISTINCT job_id) DESC) AS rank
    FROM skills_master
    GROUP BY role_group, skill_name, skill_category
)
SELECT role_group, rank, skill_name, skill_category, demand_count
FROM skill_ranks
WHERE rank <= 3
ORDER BY role_group, rank;
""")

with engine.connect() as conn:
    df_soal3 = pd.read_sql(q_soal3, conn)
display(df_soal3)

,role_group,rank,skill_name,skill_category,demand_count
0,Data Analytics & BI,1,Microsoft Excel,BI & Visualization,89
1,Data Analytics & BI,2,SQL,Programming Languages,56
2,Data Analytics & BI,2,Power BI,BI & Visualization,56
3,Data Analytics & BI,3,Statistics / Modeling,Data Science & ML,53
4,Data Engineering,1,SQL,Programming Languages,41
5,Data Engineering,2,Python,Programming Languages,32
6,Data Engineering,3,PostgreSQL,Databases & Warehousing,18
7,Data Science,1,Machine Learning,Data Science & ML,18
8,Data Science,2,Python,Programming Languages,16
9,Data Science,3,Statistics / Modeling,Data Science & ML,15


### Soal 4: Running Total & Analisis Pareto Kebutuhan Keahlian
**Pendekatan:** Menggunakan `SUM(demand_count) OVER(ORDER BY demand_count DESC)` untuk menghitung akumulasi kumulatif keahlian teknis secara nasional.

In [17]:
q_soal4 = text("""
WITH ranked_skills AS (
    SELECT 
        skill_name,
        skill_category,
        COUNT(DISTINCT job_id) AS demand_count
    FROM skills_master
    GROUP BY skill_name, skill_category
    ORDER BY demand_count DESC
    LIMIT 10
)
SELECT 
    skill_name,
    skill_category,
    demand_count,
    SUM(demand_count) OVER (ORDER BY demand_count DESC, skill_name) AS running_total,
    ROUND(SUM(demand_count) OVER (ORDER BY demand_count DESC, skill_name)::NUMERIC / 
          (SELECT COUNT(*) FROM skills_master) * 100, 2) AS cumulative_pct
FROM ranked_skills;
""")

with engine.connect() as conn:
    df_soal4 = pd.read_sql(q_soal4, conn)
display(df_soal4)

,skill_name,skill_category,demand_count,running_total,cumulative_pct
0,SQL,Programming Languages,118,118.0,11.12
1,Python,Programming Languages,100,218.0,20.55
2,Microsoft Excel,BI & Visualization,92,310.0,29.22
3,Statistics / Modeling,Data Science & ML,82,392.0,36.95
4,Power BI,BI & Visualization,69,461.0,43.45
5,Tableau,BI & Visualization,54,515.0,48.54
6,Machine Learning,Data Science & ML,47,562.0,52.97
7,AWS,Big Data & Cloud,28,590.0,55.61
8,Google Sheets,BI & Visualization,28,618.0,58.25
9,PostgreSQL,Databases & Warehousing,28,646.0,60.89


### Soal 5: Analisis Selisih Gaji terhadap Rata-rata Profesi (Window vs Group Summary)
**Pendekatan:** Menggunakan Window Function `AVG(salary_median) OVER(PARTITION BY role_group)` untuk mempertahankan baris individu sembari menghitung selisih nominalnya.

In [18]:
q_soal5 = text("""
SELECT 
    job_id,
    company_raw,
    role_group,
    salary_median,
    ROUND(AVG(salary_median) OVER (PARTITION BY role_group), 0) AS avg_role_salary,
    ROUND(salary_median - AVG(salary_median) OVER (PARTITION BY role_group), 0) AS diff_from_role_avg
FROM jobs_master
WHERE is_salary_disclosed = TRUE
ORDER BY role_group, diff_from_role_avg DESC
LIMIT 10;
""")

with engine.connect() as conn:
    df_soal5 = pd.read_sql(q_soal5, conn)
display(df_soal5)

,job_id,company_raw,role_group,salary_median,avg_role_salary,diff_from_role_avg
0,JBS_95499288,PT Adecco Personnel Indonesia,Data Analytics & BI,25000000.0,6347861.0,18652139.0
1,GLN_ddfe2b573476400aa025eb6b4668b86f,Nusantara Sakti Group,Data Analytics & BI,17500000.0,6347861.0,11152139.0
2,JBS_95615063,PT Fata Organa Solusi,Data Analytics & BI,12000000.0,6347861.0,5652139.0
3,JBS_95292658,PT Dupoin Group Indonesia,Data Analytics & BI,11250000.0,6347861.0,4902139.0
4,JBS_95523426,PT Beton Perkasa Wijaksana,Data Analytics & BI,11000000.0,6347861.0,4652139.0
5,GLN_aa07c9b62ee94dccbbbc64ff0734a98f,PT Sigma Global Teknologi,Data Analytics & BI,10000000.0,6347861.0,3652139.0
6,JBS_95492157,PT DUNIA KECANTIKAN INDONESIA,Data Analytics & BI,10000000.0,6347861.0,3652139.0
7,GLN_036638bdda7f477aacd0a310040b2fbb,PT Digital Solusi Grup,Data Analytics & BI,9750000.0,6347861.0,3402139.0
8,GLN_06ecc90733644f7096bad04eeff207f2,PT Pro Sigmaka Mandiri,Data Analytics & BI,9000000.0,6347861.0,2652139.0
9,GLN_a072678371244916879752a6bee58e07,Nusantara Sakti Group,Data Analytics & BI,8500000.0,6347861.0,2152139.0


--- 
## BAGIAN IV: TEORI FONDASI DATABASE & URUTAN EKSEKUSI
--- 
### 1. Urutan Eksekusi Query SQL (Order of Execution)
Salah satu pertanyaan interview konseptual yang paling sering diuji:

```
1. FROM       -> Mengidentifikasi tabel asal
2. ON         -> Evaluasi kondisi relasi JOIN
3. JOIN       -> Menggabungkan tabel
4. WHERE      -> Memfilter baris SEBELUM pengelompokan
5. GROUP BY   -> Mengelompokkan baris berdasarkan dimensi
6. HAVING     -> Memfilter kelompok SESUDAH agregasi
7. SELECT     -> Menentukan kolom dan menghitung fungsi (termasuk Window Functions)
8. DISTINCT   -> Menghapus duplikasi baris hasil
9. ORDER BY   -> Mengurutkan hasil akhir
10. LIMIT     -> Membatasi jumlah baris tampilan
```

> [!IMPORTANT]
> **Mengapa kolom alias pada `SELECT` tidak bisa dipakai di `WHERE`?**
> Karena klausa `WHERE` dieksekusi pada langkah ke-4, sedangkan klausa `SELECT` (di mana alias dibuat) baru dieksekusi pada langkah ke-7! Sebaliknya, alias dapat digunakan pada `ORDER BY` karena dieksekusi setelah `SELECT` (langkah ke-9).

### 2. Perbedaan `DELETE` vs `TRUNCATE` vs `DROP`
- **`DELETE`**: Perintah DML. Menghapus baris tertentu dengan kondisi `WHERE`. Lebih lambat karena mencatat *transaction log* per baris dan dapat di-rollback.
- **`TRUNCATE`**: Perintah DDL. Mengosongkan seluruh isi tabel sekaligus, mereset auto-increment identity, dan jauh lebih cepat karena tidak mencatat log per baris.
- **`DROP`**: Perintah DDL. Menghapus tabel secara permanen beserta definisi skema, struktur kolom, dan indeksnya dari basis data.

### 3. Ringkasan & Status Kesiapan Pipeline Data

Proses ingestion dan pemodelan relasional telah berhasil dituntaskan:
1. Tabel master relasional `jobs_master` dan `skills_master` terisi penuh dengan 250 lowongan dan 1.061 pemetaan keahlian teknis.
2. Integritas referensial Foreign Key terverifikasi valid (0 orphan records).
3. Seluruh konsep query analitik dari tingkat dasar hingga *Advanced Window Functions* telah berhasil dipraktikkan.
4. Basis data siap dikoneksikan ke antarmuka visualisasi dashboard Streamlit maupun alat BI seperti Metabase atau Power BI.